# XGBoost

[...]

## Importando as Bibliotecas

Primeiro vamos carregar nossas bibliotecas.

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## Carregando o Dataset

Agora vamos carregar nosso dataset.

In [2]:
df = pd.read_parquet("../../data/processed/UCMF_fitted.parquet")

In [3]:
df.info()

<class 'pandas.DataFrame'>
Index: 11705 entries, 0 to 12872
Data columns (total 30 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   peso                    9587 non-null   Float64 
 1   altura                  8078 non-null   Int64   
 2   imc                     7708 non-null   Int64   
 3   idade                   10865 non-null  Float64 
 4   pulsos                  11657 non-null  category
 5   pa_sistolica            5131 non-null   Int64   
 6   pa_diastolica           5121 non-null   Int64   
 7   ppa                     10768 non-null  category
 8   patologia               11705 non-null  category
 9   b2                      11674 non-null  category
 10  sopro                   11682 non-null  category
 11  fc                      10986 non-null  Int64   
 12  hda1                    8565 non-null   category
 13  hda2                    11705 non-null  category
 14  sexo                    11701 non-null

## Treinando o Modelo

Agora vamos treinar nosso modelo. Primeiramente iremos montar nosso pipeline com imputers, encoders, scalers e nosso modelo de Regressão Logística com validação cruzada.

In [4]:
from xgboost import XGBClassifier

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import (
    SimpleImputer,
    KNNImputer,
)
from sklearn.preprocessing import (
    OneHotEncoder,
    LabelEncoder,
)
from sklearn.preprocessing import (
    MinMaxScaler,
    MaxAbsScaler,
    StandardScaler,
    RobustScaler
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    train_test_split,
)

random_state = 42

cross_validator = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=random_state,
)

def create_pipeline(X):
    numeric_columns = X.select_dtypes(include="number").columns.to_list()
    categorical_columns = X.select_dtypes(include="category").columns.to_list()

    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="aucpr",
        use_label_encoder=False,
        random_state=random_state,
        verbosity=0,
        n_jobs=-1
    )

    numeric_pipeline = Pipeline(steps=[
        ("imputer", KNNImputer()),
        ("scaler", RobustScaler())
    ])

    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ("scaler", MaxAbsScaler())
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric_features", numeric_pipeline, numeric_columns),
            ("categorical_features", categorical_pipeline, categorical_columns),
        ],
        remainder="passthrough"
    )

    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model),
    ])

    return pipeline

label = LabelEncoder()

X = df.drop(["patologia"], axis="columns")
y = label.fit_transform(df["patologia"])

pipeline = create_pipeline(X)

Agora vamos avaliar o desempenho do nosso modelo usando validação cruzada.

In [5]:
def evaluate_pipeline(X, y):
    pipeline = create_pipeline(X)

    scores = cross_validate(
        pipeline,
        X,
        y,
        cv=cross_validator,
        scoring={
            "roc_auc": "roc_auc",
            "accuracy": "accuracy",
            "precision": "precision",
            "recall": "recall",
            "f1": "f1"
        }
    )

    return scores

In [8]:
metrics = evaluate_pipeline(X.drop("sopro", axis="columns"), y)

In [9]:
for metric, scores in metrics.items():
    print(f"{metric:14} = {scores.mean().round(2)}")

fit_time       = 19.83
score_time     = 7.21
test_roc_auc   = 0.79
test_accuracy  = 0.73
test_precision = 0.72
test_recall    = 0.62
test_f1        = 0.66


Como podemos notar, nós obtemos um modelo com $\text{ROC-AUC}$ de $93\%$, um valor bem alto para um modelo simples como a Regressão Logística. Isso nos permite testar nossa hipótese de que a feature `"sopro"` possa estar causando data leakage. Além disso iremos testar as hipóteses de que as features `"id"` e `"convenio"` não trazem benefício para nosso modelo.

In [15]:
for col in ["id", "convenio", "sopro"]:
    metrics = evaluate_pipeline(X.drop(col, axis="columns"), y)
    print(f"ROC-AUC sem {col:8} = {metrics["test_roc_auc"].mean().round(2)}")

metrics = evaluate_pipeline(X.filter(items=["sopro"], axis="columns"), y)
print(f"ROC-AUC com {"sopro":8} = {metrics["test_roc_auc"].mean().round(2)}")

ROC-AUC sem id       = 0.94
ROC-AUC sem convenio = 0.94
ROC-AUC sem sopro    = 0.79
ROC-AUC com sopro    = 0.91


Com essas métricas percebemos que remover as features `"id"` e `"convenio"` não alteram o $\text{ROC-AUC}$, o que é um forte indício de que elas não são relevantes para nosso problema. Para confirmar isso vamos rodar um teste de permutação para calcular a importância de cada feature. Vamos primeiramente ajustar nosso aos dados de treinamento.

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    train_size=0.7,
    random_state=random_state,
    stratify=y
)

pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric_features', ...), ('categorical_features', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the ou

Agora vamos executar o teste de permutação.

In [ ]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    pipeline,
    X_test,
    y_test,
    scoring="roc_auc",
    n_repeats=5,
    random_state=random_state,
    n_jobs=-1,
)

E vamos checar os resultados do teste de permutação.

In [18]:
importance = pd.DataFrame({
    "feature": X_test.columns,
    "importance": result.importances_mean,
    "std": result.importances_std
})

importance = importance.sort_values(
    "importance",
    ascending=False
)

importance

,feature,importance,std
11,sopro,0.344612,0.007682
2,altura,0.008533,0.001730
10,b2,0.008283,0.000868
17,motivo2,0.003628,0.001017
30,dias_atendimento,0.002116,0.000612
16,motivo1,0.001727,0.000812
6,pulsos,0.001356,0.000324
20,altura_faltante,0.000975,0.000528
0,id,0.000792,0.000224
22,imc_faltante,0.000786,0.000582


Como podemos ver, tanto o `"id"` quanto o `"convenio"` tem importâncias muito baixas. Isso aliado ao fato de que a ausência dessas features não muda o desempenho do nosso modelo sugere fortemente que o elas são irrelevantes. Mas como vimos, a maior parte da predição está sendo feita por causa da feature `"sopro"`, então ela esteja mascarando a real importância das demais features. Então vamos repetir calcular o desempenho do pipeline sem `"sopro"`.

In [23]:
for col in ["id", "convenio"]:
    metrics = evaluate_pipeline(X.drop([col, "sopro"], axis="columns"), y)
    print(f"ROC-AUC sem sopro e {col:8} = {metrics["test_roc_auc"].mean().round(2)}")

metrics = evaluate_pipeline(X.drop(["id", "convenio", "sopro"], axis="columns"), y)
print(f"ROC-AUC sem sopro e {"demais":8} = {metrics["test_roc_auc"].mean().round(2)}")

ROC-AUC sem sopro e id       = 0.79
ROC-AUC sem sopro e convenio = 0.79
ROC-AUC sem sopro e demais   = 0.79


Com isso fica confirmado a nossa suspeita de que ambas as features são irrelevantes. Sendo assim podemos removê-las do nosso dataset para usar posteriormente nos demais modelos. Além disso, mantemos nossa suspeita em relação à feature `"sopro"`, pois como vimos nos testes, ela é a feature com maior poder preditivo, portanto devemos estar atentos a ela nos demais modelos.

## Salvando as Alterações

Por fim, vamos salvar as mudanças que descobrimos. Basicamente vamos remover as features `"id"` e `"convenio"`.

In [24]:
df = df.drop(["id", "convenio"], axis="columns")

In [27]:
df.to_parquet("../../data/processed/UCMF_fitted.parquet")